# India Drought Analysis (State: Odisha)
## Notebook 04 (Week 5): AI tool integration – summarisation inputs, filled prompts, a rule-based baseline and a fact-checker (Odisha only)

| | |
|---|---|
| **Course** | CUTM1018 – Data Analysis and Visualization with Python |
| **Faculty** | Chandra Sekhar Jena |
| **Students** | Piyush Patnaik (250301370280), Vegulla Virendra Kaushik (250301370267) |
| **Dataset** | Drought Atlas of India 1901–2025, v2 (Chuphal, Kushwaha, Aadhar & Mishra; IIT Gandhinagar / IIT Jodhpur) – https://zenodo.org/records/20368514 |
| **Scope** | Odisha statewide series + 30 Odisha districts |
| **Week 5 deliverable** | AI comparison report (this notebook gives the tools for it) |

**Honesty note.** Up to now the only AI tool used in this project is **Grok Bot** (the assistant that built the notebooks and the dashboard).
This notebook does **not** contain any answers from ChatGPT, Gemini or Claude. It prepares everything needed to test them fairly:

1. **AI-assisted summarisation inputs** – short, real text blocks made from the clean data (`describe()`, correlation summary, key tables)
   and the course prompt types filled in with them, ready to paste (also written to `reports/AI_PROMPTS.md`).
2. **A rule-based (non-AI) summary** made directly from the data – a baseline to compare AI summaries with.
3. **A fact-check helper** – it compares numeric claims (name, value) with values computed here. We test it on the numbers
   Grok Bot wrote in `reports/ODISHA_ONLY_FACTS.md`.

This notebook needs only `data/clean/` (no raw files).

## 1. Setup and data

In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Find the project folder: walk up from the current folder until we see data/clean
# (works if Jupyter was started in the project folder or in notebooks/)
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "clean").is_dir()), Path.cwd())
if not (PROJECT / "data" / "clean").is_dir():
    raise FileNotFoundError("Cannot find the project folder (the folder that contains data/clean). Open this notebook from inside the "
                            "unzipped project folder and start Jupyter there - a single notebook file cannot run on its own. "
                            "See RUN_ON_YOUR_DEVICE.md.")
CLEAN = PROJECT / "data" / "clean"
REPORTS = PROJECT / "reports"
AI_DIR = REPORTS / "ai"
AI_DIR.mkdir(parents=True, exist_ok=True)

# Clean tables from notebook 01 (Odisha only)
odisha = pd.read_csv(CLEAN / "odisha_month.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
odisha_year = pd.read_csv(CLEAN / "odisha_year.csv").sort_values("year")
dist_month = pd.read_csv(CLEAN / "odisha_district_month.csv", parse_dates=["date"])
dist_year = pd.read_csv(CLEAN / "odisha_district_year.csv")
lookup_dist = pd.read_csv(CLEAN / "lookup_odisha_districts_clean.csv")

# Odisha district groups (same as notebooks 02-03)
GROUPS = {
    "West": ["Bargarh", "Balangir", "Boudh", "Deogarh", "Jharsuguda", "Kalahandi", "Nuapada", "Sambalpur", "Subarnapur", "Sundargarh"],
    "Coastal": ["Balasore", "Bhadrak", "Kendrapara", "Jagatsinghpur", "Puri", "Ganjam"],
    "South": ["Koraput", "Malkangiri", "Nabarangpur", "Rayagada", "Gajapati", "Kandhamal"],
    "North & Central": ["Angul", "Dhenkanal", "Keonjhar", "Mayurbhanj", "Jajpur", "Cuttack", "Khordha", "Nayagarh"],
}
district_group = {d: g for g, ds in GROUPS.items() for d in ds}
dist_month["group"] = dist_month["district"].map(district_group)
dist_year["group"] = dist_year["district"].map(district_group)
assert dist_month["group"].notna().all(), "every district must have a group"

for name, df in [("odisha_month", odisha), ("odisha_year", odisha_year),
                 ("odisha_district_month", dist_month), ("odisha_district_year", dist_year)]:
    print(f"{name:22s} {df.shape}")

odisha_month           (1500, 12)
odisha_year            (125, 17)
odisha_district_month  (45000, 15)
odisha_district_year   (3750, 19)


In [2]:
# Small helpers used all through the notebook
def share(df, y0, y1, col="drought"):
    # Percentage of months with drought (col == 1) between years y0 and y1 (inclusive)
    part = df[(df["year"] >= y0) & (df["year"] <= y1)]
    return 100 * part[col].mean()

def drought_events(flags, spei, dates):
    # Run theory: an event is a run of consecutive drought months (flag = 1).
    # Returns one row per event: start date, duration (months) and severity (sum of -SPEI over the event).
    flags = pd.Series(flags).reset_index(drop=True)
    spei = pd.Series(spei).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    run_id = (flags != flags.shift()).cumsum()           # a new id every time the flag changes
    ev = pd.DataFrame({"run": run_id, "flag": flags, "spei": spei, "date": dates})
    ev = ev[ev["flag"] == 1].groupby("run").agg(start=("date", "first"), duration=("date", "size"),
                                                  severity=("spei", lambda s: -s.sum()))
    return ev.reset_index(drop=True)

def as_text(df, decimals=2):
    # Turn a table into plain text for a prompt (rounded, no index clutter)
    return df.round(decimals).to_string()

## 2. AI-assisted summarisation: building the real input text

An AI tool can only summarise what we give it. We cannot paste 45,000 rows into a chat, so we build **compact text blocks** from the
clean data: the table structure, `describe()`, a correlation summary and a few key tables. Every number in these blocks is computed
here. The same blocks are put into the prompts in Section 2.7.

### 2.1 Dataset structure block

In [3]:
def structure_block():
    lines = ["Dataset: Drought Atlas of India 1901-2025 v2 (monthly SPEI, area averages), Odisha only. Cleaned tables used:"]
    for name, df in [("odisha_month (statewide)", odisha), ("odisha_district_month", dist_month.drop(columns="group"))]:
        lines.append(f"- {name}: {df.shape[0]} rows x {df.shape[1]} columns")
        lines.append("  columns: " + ", ".join(f"{c} ({str(t)})" for c, t in df.dtypes.items()))
    lines.append(f"- Period: {odisha['date'].min():%b %Y} to {odisha['date'].max():%b %Y}; "
                 f"1 statewide series; {dist_month['district'].nunique()} Odisha districts")
    miss = odisha[["spei_1", "spei_4", "spei_12"]].isna().sum()
    lines.append("- Missing values in odisha_month: " + ", ".join(f"{c}={v}" for c, v in miss.items())
                 + " (SPEI-4 and SPEI-12 cannot be computed for the first 3 and 11 months of 1901)")
    lines.append("- drought = 1 if spei_1 <= -1; severity classes: Moderate (-1.5 < x <= -1), Severe (-2 < x <= -1.5), Extreme (x <= -2)")
    lines.append("- Each SPEI series is standardised over 1901-2025 (mean about 0, SD about 1), so over the full period every area "
                 "has about 16% drought months.")
    return "\n".join(lines)

STRUCTURE = structure_block()
print(STRUCTURE)

Dataset: Drought Atlas of India 1901-2025 v2 (monthly SPEI, area averages), Odisha only. Cleaned tables used:
- odisha_month (statewide): 1500 rows x 12 columns
  columns: area (str), date (datetime64[us]), year (int64), month (int64), decade (int64), season (str), spei_1 (float64), spei_4 (float64), spei_12 (float64), drought (int64), severity (str), spei1_iqr_outlier (int64)
- odisha_district_month: 45000 rows x 14 columns
  columns: district_id (int64), district (str), district_name_raw (str), date (datetime64[us]), year (int64), month (int64), decade (int64), season (str), spei_1 (float64), spei_4 (float64), spei_12 (float64), drought (int64), severity (str), spei1_iqr_outlier (int64)
- Period: Jan 1901 to Dec 2025; 1 statewide series; 30 Odisha districts
- Missing values in odisha_month: spei_1=0, spei_4=3, spei_12=11 (SPEI-4 and SPEI-12 cannot be computed for the first 3 and 11 months of 1901)
- drought = 1 if spei_1 <= -1; severity classes: Moderate (-1.5 < x <= -1), Severe (-2 

### 2.2 `describe()` block

In [4]:
desc_state = odisha[["spei_1", "spei_4", "spei_12"]].describe()
desc_odisha_year = odisha_year[["spei1_mean", "drought_months", "monsoon_spei4_sep", "annual_spei12_dec"]].describe()
DESCRIBE = ("odisha_month[['spei_1','spei_4','spei_12']].describe():\n" + as_text(desc_state, 3)
            + "\n\nOdisha yearly table (125 years) describe():\n" + as_text(desc_odisha_year, 3))
print(DESCRIBE)

odisha_month[['spei_1','spei_4','spei_12']].describe():
         spei_1    spei_4   spei_12
count  1500.000  1497.000  1489.000
mean     -0.000     0.000    -0.000
std       0.996     0.996     0.996
min      -2.717    -3.146    -2.834
25%      -0.676    -0.662    -0.615
50%      -0.043     0.005     0.019
75%       0.709     0.693     0.678
max       2.728     2.683     2.926

Odisha yearly table (125 years) describe():
       spei1_mean  drought_months  monsoon_spei4_sep  annual_spei12_dec
count     125.000         125.000            125.000            125.000
mean       -0.000           1.944             -0.000              0.000
std         0.330           1.572              1.000              1.000
min        -0.875           0.000             -2.880             -2.804
25%        -0.180           1.000             -0.563             -0.540
50%         0.040           2.000              0.042              0.013
75%         0.213           3.000              0.531              0.660

### 2.3 Correlation summary block
A full 30 × 30 matrix is too long for a prompt, so we summarise it: the correlation of the districts with the statewide series, averages within/between district groups and the strongest and weakest pairs.

In [5]:
# Districts vs the statewide series (monthly SPEI-1), and between SPEI timescales for Odisha (monthly)
state_s = odisha.set_index("date")["spei_1"]
r_dist_state = pd.Series({d: stats.pearsonr(g.set_index("date")["spei_1"], state_s.loc[g["date"]])[0]
                          for d, g in dist_month.groupby("district")})
timescale_corr = odisha[["spei_1", "spei_4", "spei_12"]].corr()

# Monsoon (September SPEI-4) correlation between the 30 Odisha districts
sep = dist_month[dist_month["month"] == 9].pivot(index="year", columns="district", values="spei_4")
corr = sep.corr()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))      # each pair once
pairs = upper.rename_axis(index="district_a", columns="district_b").stack().dropna().rename("r").reset_index()
pairs["same_group"] = pairs["district_a"].map(district_group) == pairs["district_b"].map(district_group)

west, coast = GROUPS["West"], GROUPS["Coastal"]
west_coast_r = corr.loc[west, coast].values.mean()

CORRELATION = "\n".join([
    f"Odisha districts vs the statewide series, monthly SPEI-1: mean Pearson r = {r_dist_state.mean():.2f} "
    f"(lowest {r_dist_state.min():.2f} {r_dist_state.idxmin()}, highest {r_dist_state.max():.2f} {r_dist_state.idxmax()})",
    "Odisha correlation between SPEI timescales (monthly):",
    as_text(timescale_corr, 2),
    f"Odisha districts, September SPEI-4 (monsoon), {len(pairs)} district pairs:",
    f"  mean r all pairs = {pairs['r'].mean():.2f}; same group = {pairs.loc[pairs['same_group'], 'r'].mean():.2f}; "
    f"different groups = {pairs.loc[~pairs['same_group'], 'r'].mean():.2f}; West vs Coastal = {west_coast_r:.2f}",
    "  strongest pairs: " + "; ".join(f"{a}-{b} {r:.2f}" for a, b, r in pairs.nlargest(3, "r")[["district_a", "district_b", "r"]].values),
    "  weakest pairs: " + "; ".join(f"{a}-{b} {r:.2f}" for a, b, r in pairs.nsmallest(3, "r")[["district_a", "district_b", "r"]].values),
])
print(CORRELATION)

Odisha districts vs the statewide series, monthly SPEI-1: mean Pearson r = 0.84 (lowest 0.69 Malkangiri, highest 0.93 Angul)
Odisha correlation between SPEI timescales (monthly):
         spei_1  spei_4  spei_12
spei_1     1.00    0.52     0.30
spei_4     0.52    1.00     0.57
spei_12    0.30    0.57     1.00
Odisha districts, September SPEI-4 (monsoon), 435 district pairs:
  mean r all pairs = 0.48; same group = 0.65; different groups = 0.42; West vs Coastal = 0.37
  strongest pairs: Khordha-Puri 0.97; Jharsuguda-Sambalpur 0.91; Balangir-Nuapada 0.91
  weakest pairs: Gajapati-Jharsuguda -0.06; Gajapati-Sundargarh -0.02; Bargarh-Gajapati 0.04


### 2.4 Key tables block

In [6]:
# (a) Odisha drought share (%) by season and period
odisha_p = odisha.assign(period=pd.cut(odisha["year"], [1900, 1960, 1990, 2025], labels=["1901-1960", "1961-1990", "1991-2025"]))
season_table = (odisha_p.groupby(["season", "period"], observed=True)["drought"].mean().unstack() * 100)
season_table = season_table.loc[["Winter (JF)", "Pre-monsoon (MAM)", "Monsoon (JJAS)", "Post-monsoon (OND)"]]

# (b) Odisha districts: drought share early vs recent + yearly SPEI-1 trend (Mann-Kendall via Kendall tau, Sen's slope)
rows = []
for d, g in dist_year.sort_values("year").groupby("district"):
    tau, p = stats.kendalltau(g["year"], g["spei1_mean"])
    sen = stats.theilslopes(g["spei1_mean"], g["year"])[0] * 10
    dm = dist_month[dist_month["district"] == d]
    rows.append({"district": d, "group": district_group[d], "share_1901_1960": share(dm, 1901, 1960),
                 "share_1991_2025": share(dm, 1991, 2025), "kendall_tau": tau, "mk_p": p, "sen_per_decade": sen})
dist_table = pd.DataFrame(rows).set_index("district").sort_values("share_1991_2025", ascending=False)
dist_table["rank_recent"] = range(1, len(dist_table) + 1)

# (c) Odisha statewide trend
tau_od, p_od = stats.kendalltau(odisha_year["year"], odisha_year["spei1_mean"])
sen_od = stats.theilslopes(odisha_year["spei1_mean"], odisha_year["year"])[0] * 10

KEY_TABLES = "\n\n".join([
    "Odisha statewide drought share (% of months with SPEI-1 <= -1) by season and period:\n" + as_text(season_table, 1),
    "Odisha districts, top 5 and bottom 3 by drought share 1991-2025 (of 30; trend = yearly mean SPEI-1, 1901-2025):\n"
    + as_text(pd.concat([dist_table.head(5), dist_table.tail(3)]).drop(columns="rank_recent"), 3),
    "Mean drought share 1991-2025 by district group: "
    + "; ".join(f"{g} {v:.1f}%" for g, v in dist_table.groupby("group")["share_1991_2025"].mean().sort_values(ascending=False).items()),
    f"Odisha statewide drought share: {share(odisha, 1901, 1960):.1f}% in 1901-1960 and {share(odisha, 1991, 2025):.1f}% in 1991-2025",
    f"Odisha statewide yearly mean SPEI-1 trend 1901-2025: Kendall tau = {tau_od:.3f}, p = {p_od:.4f}, Sen's slope = {sen_od:.3f} per decade",
])
print(KEY_TABLES)

Odisha statewide drought share (% of months with SPEI-1 <= -1) by season and period:
period              1901-1960  1961-1990  1991-2025
season                                             
Winter (JF)               4.2       16.7       37.1
Pre-monsoon (MAM)        10.6       17.8       24.8
Monsoon (JJAS)           13.8       22.5       17.1
Post-monsoon (OND)        6.1       20.0       26.7

Odisha districts, top 5 and bottom 3 by drought share 1991-2025 (of 30; trend = yearly mean SPEI-1, 1901-2025):
                         group  share_1901_1960  share_1991_2025  kendall_tau   mk_p  sen_per_decade
district                                                                                            
Nuapada                   West            5.556           30.952       -0.358  0.000          -0.054
Nabarangpur              South            6.528           28.095       -0.282  0.000          -0.041
Balangir                  West            6.528           27.619       -0.330  0.000  

### 2.5 Real debugging cases from this project
Two problems that really happened while building the notebooks (recorded in `WORK_LOG.md` and in Grok Bot's working notes).
We reproduce them here so the debugging prompt contains real code and real output.

* **Case A – district names do not match.** The atlas names are corrupted (`|` for I, `@` for U) and some use other spellings, so a
  naive name join fails.
* **Case B – `DataFrame.round()` warning.** Rounding a table that still has a datetime column gives a `UserWarning` in pandas 3.

In [7]:
# Case A: naive join on names (raw atlas name -> title case) vs our clean district names
naive = lookup_dist.assign(naive_name=lookup_dist["district_name_raw"].str.title())
failed = naive[naive["naive_name"] != naive["district"]][["district_name_raw", "naive_name", "district"]]
print(f"Naive name join: {len(failed)} of {len(naive)} districts do not match")
CASE_A_OUTPUT = failed.to_string(index=False)
print(CASE_A_OUTPUT)

Naive name join: 12 of 30 districts do not match
   district_name_raw           naive_name    district
              ANUGUL               Anugul       Angul
 BOLANGIR (BALANGIR)  Bolangir (Balangir)    Balangir
BALASORE (BALESHWAR) Balasore (Baleshwar)    Balasore
            BARAGARH             Baragarh     Bargarh
             JAJAPUR              Jajapur      Jajpur
         KENDRAPARHA          Kendraparha  Kendrapara
KEONJHAR (KENDUJHAR) Keonjhar (Kendujhar)    Keonjhar
          MAY@RBHANJ           May@Rbhanj  Mayurbhanj
        NABARANGAPUR         Nabarangapur Nabarangpur
            N@APARHA             N@Aparha     Nuapada
           RAYAGARHA            Rayagarha    Rayagada
       BAUDH (BAUDA)        Baudh (Bauda)       Boudh


In [8]:
# Case B: round() on a frame that still has a datetime column
sample = odisha[["date", "spei_1", "spei_4"]].head(3)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sample.round(1)
CASE_B_WARNING = "; ".join(f"{w.category.__name__}: {w.message}" for w in caught)
print(CASE_B_WARNING)

### 2.6 Reference numbers for the code-generation prompt
If an AI writes a run-theory drought-event function, we need known answers to test it with.

In [9]:
od_events = drought_events(odisha["drought"], odisha["spei_1"], odisha["date"])
longest = od_events.loc[od_events["duration"].idxmax()]
CODEGEN_CHECK = (f"Odisha statewide series (SPEI-1 <= -1, Jan 1901 - Dec 2025): {len(od_events)} events; "
                 f"longest = {longest['duration']} months starting {longest['start']:%b %Y}; "
                 f"events lasting 1 month = {(od_events['duration'] == 1).sum()}")
print(CODEGEN_CHECK)

Odisha statewide series (SPEI-1 <= -1, Jan 1901 - Dec 2025): 184 events; longest = 8 months starting Jul 2000; events lasting 1 month = 145


### 2.7 The filled prompts (written to `reports/AI_PROMPTS.md`)
The prompt types follow the course guide's prompt library (dataset understanding, EDA insights, visualization suggestions, code debugging),
plus a summarisation prompt and a code-generation prompt. Each prompt asks the AI to use **only** the numbers given, so its answer can be
fact-checked with Section 4.

In [10]:
ROLE = ("You are helping two second-year B.Tech students with a university data-analysis project "
        "(course CUTM1018, Data Analysis and Visualization with Python). Project: India Drought Analysis (State: Odisha) - "
        "the study covers Odisha only: the statewide series and its 30 districts.")
RULES = ("Rules: use only the numbers and columns given below; do not invent columns or values; "
         "if something cannot be answered from the data, say so; keep the language simple.")

PROMPTS = {}
PROMPTS["P1"] = ("Dataset understanding", f"""{ROLE}
{RULES}

Here is the structure of our cleaned dataset:
{STRUCTURE}

Summary statistics:
{DESCRIBE}

Questions:
1. Explain in simple words what SPEI-1, SPEI-4 and SPEI-12 measure and what a value of -1 or -2 means.
2. Why does every area have about 16% drought months over the full 1901-2025 period? What does this mean for comparing districts?
3. List the data-quality checks we should do on these tables (missing values, duplicates, date gaps, outliers) and say which of the numbers above already answer them.
4. Which columns would you use to study (a) monsoon drought and (b) long-term trends?""")

PROMPTS["P2"] = ("EDA insights", f"""{ROLE}
{RULES}

Correlation summary:
{CORRELATION}

Key tables:
{KEY_TABLES}

Task: give 6 short insights from these numbers. For each insight write: (1) the insight in one sentence, (2) the exact numbers from the tables that support it, (3) one possible limitation. Mark any statement that is not directly supported by the numbers as "HYPOTHESIS".""")

PROMPTS["P3"] = ("Visualization suggestions", f"""{ROLE}
{RULES}

Tables and columns available:
{STRUCTURE}

Main findings so far:
{KEY_TABLES}

Task: suggest 6 charts (at least 5 different chart types) that best show these findings using Matplotlib, Seaborn, GeoPandas or Plotly. For each chart give: chart type, which table and columns, what goes on each axis, colour choice, and the one message the chart should show. We have an Odisha district boundary shapefile that joins to our tables on a numeric ID.""")

PROMPTS["P4a"] = ("Code debugging (name join)", f"""{ROLE}

We want to join our district table to a lookup table by name, but some districts do not match.

Code:
```python
lookup = pd.read_csv("data/clean/lookup_odisha_districts_clean.csv")   # columns: district_id, district_name_raw, district
lookup["naive_name"] = lookup["district_name_raw"].str.title()
failed = lookup[lookup["naive_name"] != lookup["district"]]
print(len(failed), "of", len(lookup), "do not match")
```
Output ({len(failed)} of {len(naive)} do not match):
{CASE_A_OUTPUT}

Questions: 1. Why do these names fail? 2. What is the safest way to join the tables? 3. Write a small cleaning function for the corrupted characters you can see in the output, and say which mismatches it will NOT fix.""")

PROMPTS["P4b"] = ("Code debugging (round warning)", f"""{ROLE}

Code (pandas {pd.__version__}):
```python
sample = odisha[["date", "spei_1", "spei_4"]].head(3)   # date is datetime64
sample.round(1)
```
Warning shown:
{CASE_B_WARNING}

Questions: 1. What does the warning mean? 2. Give two correct ways to round only the numeric columns. 3. Is the result wrong, or is it only a warning?""")

PROMPTS["P5"] = ("Summarisation", f"""{ROLE}
{RULES}

{STRUCTURE}

{KEY_TABLES}

Task: write a 150-word summary of what these numbers say about drought in Odisha, for a reader who is not a climate expert. Use at least 5 exact numbers from the tables. Then list every number you used in the form name = value, one per line.""")

PROMPTS["P6"] = ("Code generation", f"""{ROLE}

Table odisha_month (pandas DataFrame) has columns: area, date (datetime64, monthly), year, month, spei_1 (float), drought (1 if spei_1 <= -1 else 0).

Task: write a commented Python function drought_events(df) using pandas only that applies run theory to ONE area's monthly series:
an event is a run of consecutive months with drought == 1. Return a DataFrame with one row per event and columns start, end, duration_months, severity (sum of -spei_1 over the event), peak_spei (minimum spei_1). Keep the code simple enough for students to explain in a viva, and show how to call it for the Odisha statewide series.

We will test your function on Odisha. Do not guess the result; we already know it.""")

for pid, (kind, text) in PROMPTS.items():
    print(f"{pid:4s} {kind:32s} {len(text):5d} characters, about {len(text.split()):4d} words")

P1   Dataset understanding             3255 characters, about  422 words
P2   EDA insights                      3224 characters, about  372 words
P3   Visualization suggestions         3860 characters, about  483 words
P4a  Code debugging (name join)        1644 characters, about  178 words
P4b  Code debugging (round warning)     682 characters, about   99 words
P5   Summarisation                     3686 characters, about  460 words
P6   Code generation                    973 characters, about  154 words


In [11]:
# Write all prompts to reports/AI_PROMPTS.md (ready to paste)
PURPOSE = {
    "P1": "Does the tool understand SPEI, the table structure and the standardisation caveat?",
    "P2": "Can it find correct insights from real summary tables without inventing numbers?",
    "P3": "Are its chart ideas useful and possible with our columns?",
    "P4a": "Can it explain a real failed join (corrupted names) and suggest the safe fix?",
    "P4b": "Can it explain a real pandas warning correctly?",
    "P5": "Can it summarise accurately for a non-expert, listing the numbers it used?",
    "P6": "Can it write correct, simple run-theory code (tested against known answers)?",
}
header = f"""# AI prompts – ready to paste (Week 5, Odisha-only version)

Project: India Drought Analysis (State: Odisha) – CUTM1018. The study covers **Odisha only** (statewide series + 30 districts). Generated by `notebooks/04_ai_integration.ipynb`; every number below is computed from `data/clean/`.

The four course prompt types (dataset understanding, EDA insights, visualization suggestions, code debugging) come from the course guide's prompt library (Part 5.3); summarisation and code-generation prompts are added for Chapter 6. The guide's own wording was not available in the project folder, so the wording here is ours, filled in with this project's real columns and statistics.

## How to use
1. Open a **new chat** in each tool (ChatGPT, Gemini, Claude – and Grok Bot if you want a fourth column). Paste one prompt at a time, exactly as written (everything inside the grey box).
2. Save each full answer as text in `reports/ai/responses/<tool>_<prompt id>.md` (e.g. `chatgpt_P2.md`). Note the date and model name shown by the tool.
3. Copy every number the AI states into a claims list and run the fact-check helper (Section 4 of the notebook; the list of checkable facts is in `reports/ai/fact_catalogue.csv`).
4. For P4/P6 run the AI's code in the notebook and compare with the known answers listed under "How to check".
5. Fill in the comparison table in `reports/WEEK5_AI_COMPARISON_REPORT.md` (Section B).

| ID | Course prompt type | Purpose |
|---|---|---|
""" + "\n".join(f"| {pid} | {kind} | {PURPOSE[pid]} |" for pid, (kind, _) in PROMPTS.items()) + "\n"

checks = {
    "P1": f"Answer to Q2 must mention standardisation (mean 0, SD 1 over 1901-2025). Missing-value counts must match the structure block (spei_4 = {int(odisha['spei_4'].isna().sum())}, spei_12 = {int(odisha['spei_12'].isna().sum())} in odisha_month).",
    "P2": "Every number must appear in the tables given. Use the fact catalogue names, e.g. odisha_share_1991_2025, trend_tau_spei1_mean, season_winter_1991_2025, share_1991_2025_Nuapada.",
    "P3": "Judge usefulness: does each chart match an existing column? Did it suggest a district map joined on ID (not names)? Did it invent variables (e.g. rainfall in mm, temperature) that the dataset does not have?",
    "P4a": f"A correct answer joins on district_id, not names. Known: {len(failed)} of {len(naive)} names fail; the corrupted character in the Odisha names is @ (for U); other failures are spelling differences (e.g. BOLANGIR (BALANGIR)).",
    "P4b": "A correct answer says it is only a warning (numeric columns are still rounded) and rounds numeric columns only, e.g. df.round({'spei_1': 1, 'spei_4': 1}) or select_dtypes('number').",
    "P5": "Fact-check every 'name = value' line. Penalise any number that is not in the tables.",
    "P6": "Run the function on Odisha. Known answer: " + CODEGEN_CHECK + ".",
}
parts = [header]
for pid, (kind, text) in PROMPTS.items():
    parts.append(f"\n## {pid} – {kind}\n\n```text\n{text}\n```\n\n**How to check:** {checks[pid]}\n")
(REPORTS / "AI_PROMPTS.md").write_text("".join(parts))
print("wrote reports/AI_PROMPTS.md", f"({(REPORTS / 'AI_PROMPTS.md').stat().st_size / 1000:.1f} kB)")

wrote reports/AI_PROMPTS.md (21.3 kB)


**What this shows:** The input blocks are short (the longest prompt is about 480 words) but carry real numbers: 1500 statewide months, 45,000 district-months, 243 statewide drought months, and the real debugging cases (12 of 30 district names fail a naive join; the statewide round() warning). The code-generation check is also fixed in advance: 184 statewide events, the longest 8 months from July 2000, 145 events of one month. Because the prompts say 'use only these numbers', an AI answer can be checked line by line.

## 3. Rule-based (non-AI) summary – the baseline

This function writes sentences using fixed rules (if/else on computed numbers). It cannot "hallucinate": every number comes from the data.
It is a baseline: an AI summary should be *at least* as accurate as this, and should add explanation that rules cannot.

In [12]:
def rule_based_summary():
    s = []   # list of sentences
    total = len(odisha)
    n_dr = int(odisha["drought"].sum())
    s.append(f"Odisha had {n_dr} drought months out of {total} ({100 * n_dr / total:.1f}%) between 1901 and 2025.")

    early, recent = share(odisha, 1901, 1960), share(odisha, 1991, 2025)
    word = "rose" if recent > early else "fell"
    s.append(f"The share of drought months {word} from {early:.1f}% in 1901-1960 to {recent:.1f}% in 1991-2025 "
             f"({recent - early:+.1f} percentage points).")

    if p_od < 0.05:
        direction = "drying" if tau_od < 0 else "wetting"
        s.append(f"The yearly mean SPEI-1 shows a significant {direction} trend (Kendall tau = {tau_od:.2f}, p = {p_od:.4f}, "
                 f"Sen's slope {sen_od:.3f} per decade).")
    else:
        s.append(f"The yearly mean SPEI-1 shows no significant trend (p = {p_od:.2f}).")

    change = season_table["1991-2025"] - season_table["1901-1960"]
    s.append(f"The biggest change is in the {change.idxmax()} season ({season_table.loc[change.idxmax(), '1901-1960']:.1f}% -> "
             f"{season_table.loc[change.idxmax(), '1991-2025']:.1f}%) and the smallest in the {change.idxmin()} season "
             f"({change.min():+.1f} points).")

    top3 = dist_table.head(3)
    s.append("Districts with the most drought months in 1991-2025: "
             + ", ".join(f"{d} ({v:.1f}%)" for d, v in top3["share_1991_2025"].items()) + ".")
    bottom = dist_table.tail(1)
    s.append(f"The district with the fewest is {bottom.index[0]} ({bottom['share_1991_2025'].iloc[0]:.1f}%); "
             f"all {int((dist_table['share_1991_2025'] > 15.87).sum())} of {len(dist_table)} districts are above the 15.9% normal level.")
    n_sig = int(((dist_table["mk_p"] < 0.05) & (dist_table["kendall_tau"] < 0)).sum())
    s.append(f"{n_sig} of {len(dist_table)} districts have a significant drying trend.")

    g = dist_table.groupby("group")["share_1991_2025"].mean()
    s.append(f"Group averages for 1991-2025: " + ", ".join(f"{k} {v:.1f}%" for k, v in g.sort_values(ascending=False).items()) + ".")

    worst = odisha_year.nsmallest(3, "monsoon_spei4_sep")
    s.append("Driest monsoon seasons (September SPEI-4): "
             + ", ".join(f"{int(y)} ({v:.2f})" for y, v in worst[["year", "monsoon_spei4_sep"]].values) + ".")
    s.append(f"The districts follow the statewide series closely but not perfectly (mean r = {r_dist_state.mean():.2f} for monthly SPEI-1, "
             f"lowest {r_dist_state.min():.2f} for {r_dist_state.idxmin()}).")
    return s

baseline = rule_based_summary()
for i, sentence in enumerate(baseline, 1):
    print(f"{i:2d}. {sentence}")

 1. Odisha had 243 drought months out of 1500 (16.2%) between 1901 and 2025.
 2. The share of drought months rose from 9.4% in 1901-1960 to 24.8% in 1991-2025 (+15.3 percentage points).
 3. The yearly mean SPEI-1 shows a significant drying trend (Kendall tau = -0.22, p = 0.0003, Sen's slope -0.028 per decade).
 4. The biggest change is in the Winter (JF) season (4.2% -> 37.1%) and the smallest in the Monsoon (JJAS) season (+3.4 points).
 5. Districts with the most drought months in 1991-2025: Nuapada (31.0%), Nabarangpur (28.1%), Balangir (27.6%).
 6. The district with the fewest is Balasore (19.0%); all 30 of 30 districts are above the 15.9% normal level.
 7. 19 of 30 districts have a significant drying trend.
 8. Group averages for 1991-2025: West 26.1%, South 25.7%, North & Central 21.8%, Coastal 21.7%.
 9. Driest monsoon seasons (September SPEI-4): 1974 (-2.88), 1987 (-2.56), 1979 (-2.17).
10. The districts follow the statewide series closely but not perfectly (mean r = 0.84 for mo

**What this shows:** The rule-based summary has 10 sentences, every number from the data. It reports 243 drought months of 1500 (16.2%), the rise from 9.4% to 24.8% between 1901–1960 and 1991–2025, a significant drying trend (tau = -0.22, p = 0.0003), Nuapada (31.0%) as the district with most drought months and Balasore (19.0%) as the least, and 19 of 30 districts with a significant drying trend. It cannot explain *why* (for example the winter jump from 4.2% to 37.1%), which is where a good AI summary may add value; but it is a safe floor for accuracy.

## 4. Fact-check helper

### 4.1 Fact catalogue: values computed in this notebook
Each fact has a short name. Every value is computed again here **from the clean monthly/yearly tables only** (not from the
notebook 02 result tables), so the check is independent of the script that wrote `reports/ODISHA_ONLY_FACTS.md`. The catalogue is saved as `reports/ai/fact_catalogue.csv`
so students can look up fact names when they check an AI answer.

In [13]:
FACTS = {}   # name -> (value, description)
def add(name, value, description):
    FACTS[name] = (float(value), description)

od_year_i = odisha_year.set_index("year")

# --- table shapes
add("rows_odisha_month", len(odisha), "rows in odisha_month.csv")
add("cols_odisha_month", odisha.shape[1], "columns in odisha_month.csv")
add("rows_odisha_district_month", len(dist_month), "rows in odisha_district_month.csv")
add("cols_odisha_district_month", dist_month.shape[1] - 1, "columns in odisha_district_month.csv (without the group column added here)")
add("n_districts", dist_month["district"].nunique(), "number of Odisha districts")
add("odisha_missing_spei4", odisha["spei_4"].isna().sum(), "missing SPEI-4 values, statewide")
add("odisha_missing_spei12", odisha["spei_12"].isna().sum(), "missing SPEI-12 values, statewide")

def iqr_outliers(x):
    q1, q3 = x.quantile([0.25, 0.75])
    return int(((x < q1 - 1.5 * (q3 - q1)) | (x > q3 + 1.5 * (q3 - q1))).sum())
add("odisha_iqr_outliers_spei1", iqr_outliers(odisha["spei_1"]), "IQR outliers in statewide SPEI-1")
add("district_iqr_outliers_spei1", iqr_outliers(dist_month["spei_1"]), "IQR outliers in district SPEI-1")

# --- Odisha statewide series
add("odisha_drought_months", odisha["drought"].sum(), "Odisha drought months (SPEI-1 <= -1), 1901-2025")
add("odisha_drought_share_pct", 100 * odisha["drought"].mean(), "Odisha drought months, % of all months 1901-2025")
add("odisha_severe_or_worse_months", (odisha["spei_1"] <= -1.5).sum(), "Odisha months with SPEI-1 <= -1.5")
add("odisha_extreme_months", (odisha["spei_1"] <= -2).sum(), "Odisha months with SPEI-1 <= -2")
add("odisha_share_1901_1960", share(odisha, 1901, 1960), "Odisha drought share %, 1901-1960")
add("odisha_share_1961_2025", share(odisha, 1961, 2025), "Odisha drought share %, 1961-2025")
add("odisha_share_1991_2025", share(odisha, 1991, 2025), "Odisha drought share %, 1991-2025")
add("odisha_change_pp", share(odisha, 1991, 2025) - share(odisha, 1901, 1960), "Odisha change in drought share, 1991-2025 minus 1901-1960 (points)")
add("odisha_drought_months_1901_1960", odisha.loc[odisha["year"] <= 1960, "drought"].sum(), "Odisha drought months 1901-1960")
add("odisha_drought_months_1991_2025", odisha.loc[odisha["year"] >= 1991, "drought"].sum(), "Odisha drought months 1991-2025")
two = odisha_p[odisha_p["period"] != "1961-1990"]
add("chi2_period", stats.chi2_contingency(pd.crosstab(two["period"].astype(str), two["drought"]))[0],
    "chi-square, Odisha period (1901-1960 vs 1991-2025) x drought month")
for dec, g in odisha.groupby("decade"):
    add(f"decade_share_{dec}", 100 * g["drought"].mean(), f"Odisha drought share %, {dec}s")
add("odisha_2000s_drought_months", odisha.loc[odisha["decade"] == 2000, "drought"].sum(), "Odisha drought months in the 2000s")
add("odisha_2010s_drought_months", odisha.loc[odisha["decade"] == 2010, "drought"].sum(), "Odisha drought months in the 2010s")
add("odisha_2000s_district_month_drought_pct", 100 * dist_month.loc[dist_month["decade"] == 2000, "drought"].mean(),
    "% of Odisha district-months in drought in the 2000s")
add("odisha_2000s_district_month_extreme_pct", 100 * (dist_month.loc[dist_month["decade"] == 2000, "spei_1"] <= -2).mean(),
    "% of Odisha district-months extreme (SPEI-1 <= -2) in the 2000s")
GKEY = {"West": "west", "Coastal": "coastal", "South": "south", "North & Central": "nc"}
for g, ds in GROUPS.items():
    add(f"group_2000s_share_{GKEY[g]}", share(dist_month[dist_month["district"].isin(ds)], 2000, 2009), f"{g} drought share %, 2000s")

# --- seasons
for s_name, k in [("Winter (JF)", "winter"), ("Pre-monsoon (MAM)", "premonsoon"), ("Monsoon (JJAS)", "monsoon"), ("Post-monsoon (OND)", "postmonsoon")]:
    for y0, y1 in [(1901, 1960), (1991, 2025)]:
        add(f"season_{k}_{y0}_{y1}", share(odisha[odisha["season"] == s_name], y0, y1), f"Odisha drought share % in {s_name}, {y0}-{y1}")
odisha_part = odisha.assign(part=np.where(odisha["month"].between(6, 9), "monsoon", "dry"))
for part_name, fact in [("dry", "odisha_dry_season_change"), ("monsoon", "odisha_monsoon_change")]:
    sub = odisha_part[odisha_part["part"] == part_name]
    add(fact, share(sub, 1991, 2025) - share(sub, 1901, 1960), f"Odisha change in {part_name}-season drought share (points)")

# --- statewide trends (Mann-Kendall = Kendall tau vs year; Sen's slope per decade)
for tag, col in [("spei1_mean", "spei1_mean"), ("monsoon", "monsoon_spei4_sep"), ("annual", "annual_spei12_dec"), ("dm", "drought_months")]:
    tau, p = stats.kendalltau(odisha_year["year"], odisha_year[col])
    add(f"trend_tau_{tag}", tau, f"Kendall tau, Odisha {col} vs year")
    add(f"trend_p_{tag}", p, f"Mann-Kendall p, Odisha {col}")
    add(f"trend_sen_{tag}", stats.theilslopes(odisha_year[col], odisha_year["year"])[0] * 10, f"Sen's slope per decade, Odisha {col}")

# --- worst monsoons and years
for y in [1974, 1987, 1979, 1965, 2002, 1924, 1901, 1998, 1996, 2000, 1972]:
    add(f"sep_spei4_{y}", od_year_i.loc[y, "monsoon_spei4_sep"], f"Odisha September SPEI-4 in {y}")
add("odisha_monsoon_drought_years", (od_year_i["monsoon_spei4_sep"] <= -1).sum(), "years with Odisha September SPEI-4 <= -1")
add("odisha_annual_drought_years", (od_year_i["annual_spei12_dec"] <= -1).sum(), "years with Odisha December SPEI-12 <= -1")
for y in [1974, 1979]:
    add(f"dec_spei12_{y}", od_year_i.loc[y, "annual_spei12_dec"], f"Odisha December SPEI-12 in {y}")
add("odisha_min_spei1", odisha["spei_1"].min(), "lowest Odisha statewide SPEI-1")
mean1 = od_year_i["spei1_mean"]
add("stripe_years_1901_1960", (mean1.loc[:1960] < -0.3).sum(), "Odisha years with mean SPEI-1 < -0.3, 1901-1960")
add("stripe_years_1961_1990", (mean1.loc[1961:1990] < -0.3).sum(), "Odisha years with mean SPEI-1 < -0.3, 1961-1990")
add("stripe_years_1991_2025", (mean1.loc[1991:] < -0.3).sum(), "Odisha years with mean SPEI-1 < -0.3, 1991-2025")

# --- events (run theory)
add("odisha_events_spei1", len(od_events), "Odisha SPEI-1 drought events (run theory), 1901-2025")
add("odisha_longest_event", od_events["duration"].max(), "longest Odisha SPEI-1 event (months)")
add("odisha_events_1_month", (od_events["duration"] == 1).sum(), "Odisha events lasting one month")
rec_ev = od_events[od_events["start"].dt.year >= 1991]
add("odisha_events_1991_2025", len(rec_ev), "Odisha events starting 1991-2025")
add("odisha_total_severity_1991_2025", rec_ev["severity"].sum(), "Odisha total event severity, events starting 1991-2025")

In [14]:
# --- district ranking (1991-2025), recomputed from the monthly table only
dist_ev = {}
for d, g in dist_month.sort_values("date").groupby("district"):
    ev = drought_events(g["drought"], g["spei_1"], g["date"])
    dist_ev[d] = ev[ev["start"].dt.year >= 1991]
rk = pd.DataFrame({
    "share_recent": {d: share(dist_month[dist_month["district"] == d], 1991, 2025) for d in dist_ev},
    "share_early": {d: share(dist_month[dist_month["district"] == d], 1901, 1960) for d in dist_ev},
    "severity": {d: e["severity"].sum() for d, e in dist_ev.items()},
    "longest": {d: e["duration"].max() for d, e in dist_ev.items()},
})
rk["change"] = rk["share_recent"] - rk["share_early"]
score = pd.concat([rk["share_recent"].rank(ascending=False, method="min"), rk["severity"].rank(ascending=False, method="min"),
                   rk["longest"].rank(ascending=False, method="min")], axis=1).mean(axis=1)
rk["composite_rank"] = score.rank(method="min")
for d, r in rk.iterrows():
    add(f"rank_{d}", r["composite_rank"], f"{d} composite rank, 1991-2025")
    add(f"share_1991_2025_{d}", r["share_recent"], f"{d} drought share %, 1991-2025")
    add(f"share_1901_1960_{d}", r["share_early"], f"{d} drought share %, 1901-1960")
    add(f"change_pp_{d}", r["change"], f"{d} change in drought share (points)")
    add(f"severity_{d}", r["severity"], f"{d} total event severity, events starting 1991-2025")
    add(f"longest_{d}", r["longest"], f"{d} longest event (months), events starting 1991-2025")
add("n_districts_more_drought", (rk["change"] > 0).sum(), "districts with a higher drought share in 1991-2025 than 1901-1960")
add("spearman_district_early_vs_recent", stats.spearmanr(rk["share_early"], rk["share_recent"])[0], "Spearman r between district shares 1901-1960 and 1991-2025")
add("spearman_share_severity", stats.spearmanr(rk["share_recent"], rk["severity"])[0], "Spearman r between 1991-2025 share and total severity")

# --- district trends (yearly mean SPEI-1, Mann-Kendall + Sen)
tr = {}
for d, g in dist_year.sort_values("year").groupby("district"):
    tau, p = stats.kendalltau(g["year"], g["spei1_mean"])
    tr[d] = {"tau": tau, "p": p, "sen": stats.theilslopes(g["spei1_mean"], g["year"])[0] * 10}
tr = pd.DataFrame(tr).T
sig = (tr["p"] < 0.05) & (tr["tau"] < 0)
add("n_districts_sig_drying", sig.sum(), "Odisha districts with significant drying (MK p < 0.05), yearly mean SPEI-1")
add("n_districts_sig_wetting", ((tr["p"] < 0.05) & (tr["tau"] > 0)).sum(), "districts with significant wetting")
for g, ds in GROUPS.items():
    add(f"sig_drying_{GKEY[g]}", sig.loc[ds].sum(), f"significant drying districts in {g}")
    add(f"mean_sen_{GKEY[g]}", tr.loc[ds, "sen"].mean(), f"mean Sen's slope per decade, {g}")
add("sen_Nuapada", tr.loc["Nuapada", "sen"], "Nuapada Sen's slope per decade")
add("tau_Nuapada", tr.loc["Nuapada", "tau"], "Nuapada Kendall tau")
for col, fact in [("monsoon_spei4_sep", "n_districts_sig_drying_monsoon"), ("annual_spei12_dec", "n_districts_sig_drying_annual")]:
    n = 0
    for d, g in dist_year.groupby("district"):
        tau, p = stats.kendalltau(g["year"], g[col])
        n += int(p < 0.05 and tau < 0)
    add(fact, n, f"districts with a significant drying trend in {col}")

# --- West vs Coastal
wk = rk.loc[GROUPS["West"]]
ck = rk.loc[GROUPS["Coastal"]]
for col, k, label in [("share_recent", "share", "drought share 1991-2025"), ("change", "change", "change since 1901-1960"), ("severity", "severity", "total severity")]:
    add(f"west_mean_{k}", wk[col].mean(), f"West mean {label}")
    add(f"coastal_mean_{k}", ck[col].mean(), f"Coastal mean {label}")
    add(f"west_coastal_welch_p_{k}", stats.ttest_ind(wk[col], ck[col], equal_var=False).pvalue, f"Welch p, West vs Coastal {label}")
    add(f"west_coastal_mw_p_{k}", stats.mannwhitneyu(wk[col], ck[col], alternative="two-sided").pvalue, f"Mann-Whitney p, West vs Coastal {label}")
add("west_mean_share_1901_1960", wk["share_early"].mean(), "West mean drought share %, 1901-1960")
add("coastal_mean_share_1901_1960", ck["share_early"].mean(), "Coastal mean drought share %, 1901-1960")
add("south_mean_share_1991_2025", rk.loc[GROUPS["South"], "share_recent"].mean(), "South mean drought share %, 1991-2025")
add("nc_mean_share_1991_2025", rk.loc[GROUPS["North & Central"], "share_recent"].mean(), "North & Central mean drought share %, 1991-2025")
wc = dist_month[(dist_month["year"] >= 1991) & dist_month["group"].isin(["West", "Coastal"])]
add("chi2_west_coastal", stats.chi2_contingency(pd.crosstab(wc["group"], wc["drought"]))[0], "chi-square, West vs Coastal district-months 1991-2025")

# --- districts vs statewide, district-district correlation
add("coherence_mean_r", r_dist_state.mean(), "mean r of district SPEI-1 with statewide SPEI-1")
add("coherence_min_r", r_dist_state.min(), "lowest district r with statewide SPEI-1")
add("coherence_max_r", r_dist_state.max(), "highest district r with statewide SPEI-1")
pooled = dist_month.merge(odisha[["date", "drought"]].rename(columns={"drought": "state_dr"}), on="date")
add("pct_district_drought_also_state", 100 * ((pooled["drought"] == 1) & (pooled["state_dr"] == 1)).sum() / (pooled["drought"] == 1).sum(),
    "% of district drought months that are also statewide drought months")
n_in = dist_month.groupby("date")["drought"].sum()
add("mean_districts_in_state_drought", n_in[odisha.set_index("date")["drought"] == 1].mean(), "mean number of districts in drought during a statewide drought month")
add("r_spei1_spei4", timescale_corr.loc["spei_1", "spei_4"], "r, statewide SPEI-1 vs SPEI-4")
add("r_spei4_spei12", timescale_corr.loc["spei_4", "spei_12"], "r, statewide SPEI-4 vs SPEI-12")
add("r_spei1_spei12", timescale_corr.loc["spei_1", "spei_12"], "r, statewide SPEI-1 vs SPEI-12")
add("district_corr_mean_all", pairs["r"].mean(), "mean r over 435 district pairs (Sep SPEI-4)")
add("district_corr_same_group", pairs.loc[pairs["same_group"], "r"].mean(), "mean r, same-group pairs")
add("district_corr_diff_group", pairs.loc[~pairs["same_group"], "r"].mean(), "mean r, different-group pairs")
add("district_corr_west_coastal", west_coast_r, "mean r, West vs Coastal")
add("district_corr_khordha_puri", corr.loc["Khordha", "Puri"], "r, Khordha vs Puri")
add("district_corr_jharsuguda_gajapati", corr.loc["Jharsuguda", "Gajapati"], "r, Jharsuguda vs Gajapati")

catalogue = pd.DataFrame([(k, v, d) for k, (v, d) in FACTS.items()], columns=["fact", "value", "description"])
catalogue.to_csv(AI_DIR / "fact_catalogue.csv", index=False)
print(len(catalogue), "facts computed; saved reports/ai/fact_catalogue.csv")
catalogue.head(10)

315 facts computed; saved reports/ai/fact_catalogue.csv


,fact,value,description
0,rows_odisha_month,1500.0,rows in odisha_month.csv
1,cols_odisha_month,12.0,columns in odisha_month.csv
2,rows_odisha_district_month,45000.0,rows in odisha_district_month.csv
3,cols_odisha_district_month,14.0,columns in odisha_district_month.csv (without ...
4,n_districts,30.0,number of Odisha districts
5,odisha_missing_spei4,3.0,"missing SPEI-4 values, statewide"
6,odisha_missing_spei12,11.0,"missing SPEI-12 values, statewide"
7,odisha_iqr_outliers_spei1,0.0,IQR outliers in statewide SPEI-1
8,district_iqr_outliers_spei1,29.0,IQR outliers in district SPEI-1
9,odisha_drought_months,243.0,"Odisha drought months (SPEI-1 <= -1), 1901-2025"


### 4.2 The helper functions
* `tolerance_from(text)` – a claim written as `24.8` is correct if the true value **rounds** to 24.8, so the tolerance is half of the
  last decimal place (0.05). An integer like `243` must match within 0.5.
* `fact_check(claims)` – takes a list of `(fact name, claimed value)` pairs (optionally with a source and a quoted text) and returns a
  table with the computed value, the difference and MATCH / MISMATCH / UNKNOWN FACT.
* `quote_found(source, quote)` – checks that the quoted text really is in the source document, so the claims are taken from the document
  and not typed again by hand.

In [15]:
def tolerance_from(text):
    # Half of the last decimal place written in the claim, e.g. "24.8" -> 0.05, "0.0003" -> 0.00005, "243" -> 0.5
    text = str(text).strip()
    decimals = len(text.split(".")[1]) if "." in text else 0
    return 0.5 * 10 ** (-decimals) + 1e-9   # tiny extra for floating-point error

def fact_check(claims, facts=FACTS):
    # claims: list of dicts with keys fact, claimed (string), and optional source, quote
    rows = []
    for c in claims:
        claimed_text = str(c["claimed"]).replace("−", "-").replace("+", "")
        claimed = float(claimed_text)
        if c["fact"] not in facts:
            rows.append({**c, "computed": np.nan, "difference": np.nan, "tolerance": np.nan, "result": "UNKNOWN FACT"})
            continue
        computed = facts[c["fact"]][0]
        tol = c.get("tolerance", tolerance_from(claimed_text))
        diff = claimed - computed
        rows.append({**c, "computed": computed, "difference": diff, "tolerance": tol,
                     "result": "MATCH" if abs(diff) <= tol else "MISMATCH"})
    return pd.DataFrame(rows)

def source_text(path):
    # Text of a Markdown file, or of all markdown cells of a notebook
    path = PROJECT / path
    if path.suffix == ".ipynb":
        import json
        nb = json.loads(path.read_text())
        return "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "markdown")
    return path.read_text()

_cache = {}
def quote_found(source, quote):
    # Whitespace is normalised, so a line break in the source does not matter
    if source not in _cache:
        _cache[source] = " ".join(source_text(source).split())
    return " ".join(quote.split()) in _cache[source]

### 4.3 Checking Grok Bot's numbers in `reports/ODISHA_ONLY_FACTS.md`
That file was written by `scripts/make_odisha_facts.py` (Grok Bot) from the notebook 02 result tables. Every table row has a description, a
value and a fact name. Here we read each row **from the file itself** (so nothing is typed again by hand) and check its value against the
catalogue above, which was computed independently from the monthly tables.

In [16]:
SRC = "reports/ODISHA_ONLY_FACTS.md"
row_pattern = re.compile(r"^\| (?P<desc>.+?) \| (?P<value>[−+\-]?\d+(?:\.\d+)?) \| `(?P<fact>[^`]+)` \|$")
doc_claims = []
for line in source_text(SRC).splitlines():
    m = row_pattern.match(line.strip())
    if m:
        doc_claims.append({"source": SRC, "quote": line.strip(), "fact": m["fact"], "claimed": m["value"], "desc": m["desc"]})
print("Claims read from", SRC, ":", len(doc_claims))

check = fact_check(doc_claims)
check["quote_found"] = [quote_found(s, q) for s, q in zip(check["source"], check["quote"])]
print("Quotes found in the source document:", int(check["quote_found"].sum()), "of", len(check))
print(check["result"].value_counts().to_string())

Claims read from reports/ODISHA_ONLY_FACTS.md : 265
Quotes found in the source document: 265 of 265
result
MATCH    265


In [17]:
# Full table (rounded for display) - the largest differences first
show = check[["fact", "claimed", "computed", "difference", "tolerance", "result", "quote_found"]].copy()
show["abs_diff_vs_tol"] = (check["difference"].abs() / check["tolerance"]).round(2)
print("Closest to the tolerance limit (abs difference / tolerance; 1.0 = at the limit):")
print(show.sort_values("abs_diff_vs_tol", ascending=False).head(12).to_string(index=False))
check.to_csv(AI_DIR / "fact_check_odisha_only_facts.csv", index=False)
print("\nsaved reports/ai/fact_check_odisha_only_facts.csv")

Closest to the tolerance limit (abs difference / tolerance; 1.0 = at the limit):
                           fact claimed   computed  difference  tolerance result  quote_found  abs_diff_vs_tol
                 sep_spei4_2002   −1.82  -1.825000    0.005000     0.0050  MATCH         True             1.00
        share_1901_1960_Bhadrak    11.2  11.250000   -0.050000     0.0500  MATCH         True             1.00
       season_monsoon_1901_1960    13.8  13.750000    0.050000     0.0500  MATCH         True             1.00
            change_pp_Sambalpur    16.2  16.250000   -0.050000     0.0500  MATCH         True             1.00
     share_1901_1960_Jharsuguda     8.8   8.750000    0.050000     0.0500  MATCH         True             1.00
        share_1901_1960_Bargarh     6.2   6.250000   -0.050000     0.0500  MATCH         True             1.00
      share_1901_1960_Sambalpur     8.8   8.750000    0.050000     0.0500  MATCH         True             1.00
     district_corr_khordha_puri

In [18]:
# Any problem?
problems = check[(check["result"] != "MATCH") | (~check["quote_found"])]
print("Claims that need attention:", len(problems))
# Does the catalogue contain facts that are not written in the document? (information only)
print("Facts in the catalogue but not in the document:", len(set(FACTS) - set(check["fact"])))
problems[["source", "quote", "fact", "claimed", "computed", "result", "quote_found"]]

Claims that need attention: 0
Facts in the catalogue but not in the document: 50


,source,quote,fact,claimed,computed,result,quote_found


**What this shows:** All 265 facts written in `reports/ODISHA_ONLY_FACTS.md` were read from the file itself and checked against values recomputed here from the monthly tables only: **265 MATCH, 0 MISMATCH, 0 UNKNOWN**, and every quoted row was found in the file. Several values sit exactly at the rounding limit (ratio 1.00, for example 11.25 written as 11.2), which is allowed because the tolerance is half of the last written decimal. 50 extra facts are in the catalogue only (for checking AI answers).

### 4.4 The helper does catch errors
**A real example.** The first time this check was run (Week 5), notebook `02_full_eda.ipynb` said Odisha's total event severity was **165.1**. The true value is
165.049…, which rounds to **165.0**. The notebook text had been written from a table
printed with two decimals (165.05) and rounded a second time. The text was corrected. Here is that old claim run through the helper:

In [19]:
old_claim = [{"fact": "odisha_total_severity_1991_2025", "claimed": "165.1", "note": "notebook 02 text before the Week 5 correction"}]
fact_check(old_claim)[["fact", "claimed", "computed", "difference", "tolerance", "result", "note"]]

,fact,claimed,computed,difference,tolerance,result,note
0,odisha_total_severity_1991_2025,165.1,165.049,0.051,0.05,MISMATCH,notebook 02 text before the Week 5 correction


**A negative control for the Odisha-only check.** We copy the document claims, change a few values on purpose (as a careless AI might), and
check that the helper reports MISMATCH for exactly those claims.

In [20]:
wrong = {"odisha_drought_months": "250", "share_1991_2025_Nuapada": "21.0", "n_districts_sig_drying": "25",
         "west_coastal_mw_p_share": "0.0500", "rank_Balasore": "1"}
control = [{**c, "claimed": wrong.get(c["fact"], c["claimed"])} for c in doc_claims]
control_result = fact_check(control)
caught = control_result[control_result["result"] == "MISMATCH"]
print("Deliberately wrong claims:", len(wrong), "| reported as MISMATCH:", len(caught), "| MATCH for all the others:",
      int((control_result["result"] == "MATCH").sum()), "of", len(control_result) - len(wrong))
caught[["fact", "claimed", "computed", "result"]]

Deliberately wrong claims: 5 | reported as MISMATCH: 5 | MATCH for all the others: 260 of 260


,fact,claimed,computed,result
9,odisha_drought_months,250,243.000000,MISMATCH
88,share_1991_2025_Nuapada,21.0,30.952381,MISMATCH
203,rank_Balasore,1,30.000000,MISMATCH
220,n_districts_sig_drying,25,19.000000,MISMATCH
237,west_coastal_mw_p_share,0.0500,0.003335,MISMATCH


### 4.5 Template: checking an answer from ChatGPT, Gemini or Claude
After pasting a prompt from `reports/AI_PROMPTS.md`, copy every number the AI gives into the list below (use the fact names in
`reports/ai/fact_catalogue.csv`) and run the cell. **The list is empty on purpose**: no other AI tool has been run yet, and we do not
make up answers.

In [21]:
ai_tool = ""          # e.g. "ChatGPT (model name, date)"
prompt_id = ""        # e.g. "P2"
ai_claims = [
    # {"fact": "odisha_share_1991_2025", "claimed": "24.8"},   # <- example format only, not an AI answer
]
if ai_claims:
    result = fact_check(ai_claims)
    print(ai_tool, prompt_id, "-", (result["result"] == "MATCH").sum(), "of", len(result), "claims match")
    display(result)
else:
    print("No AI claims entered yet - run a prompt from reports/AI_PROMPTS.md and fill in ai_claims.")

No AI claims entered yet - run a prompt from reports/AI_PROMPTS.md and fill in ai_claims.


## 5. Summary

* **Inputs for AI summarisation:** structure, describe(), correlation summary and key tables, all Odisha-only and all computed from `data/clean/`; seven prompts written to `reports/AI_PROMPTS.md`.
* **Baseline:** a 10-sentence rule-based summary with no invented numbers.
* **Fact-check helper:** catalogue of 315 facts; the 265 numbers in `reports/ODISHA_ONLY_FACTS.md` all match (265/265), and a negative control shows that 5 deliberately wrong claims are caught (5/5).
* **Not done yet (students):** running the prompts in ChatGPT, Gemini and Claude and filling the comparison report. This notebook makes no claim about those tools.

By Grok Bot (assistant) with Piyush Patnaik; no other AI tool has been used.